# Day 45：第七周综合测验 & 低代码 Workflow DSL 引擎（Dify/Coze 风格）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 45）！今天我们将亲手拆解 Dify/Coze 可视化工作流画布背后的核心原理——**声明式 Workflow DSL 解析、Kahn 拓扑环检测、变量池插值（`{{sys.query}}`）、以及 `StartNode -> KnowledgeRetrievalNode -> ConditionNode -> LLMNode -> EndNode` 条件路由 DAG 执行引擎**（`v0.4.0-day45`）。

## 1. 环境初始化与导入 `project_iterations/day_45` 模块

In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day45_dir = repo_root / "project_iterations" / "day_45"
if str(day45_dir) not in sys.path:
    sys.path.insert(0, str(day45_dir))

from zhishu_agent import (
    WorkflowEngine,
    WorkflowValidationError,
    build_default_rag_workflow_dsl,
    parse_workflow_dsl,
    render_template,
    resolve_variable,
)

print("已加载模块路径:", day45_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_45


## 2. 变量池（Variable Pool）与 `{{sys.query}}` 模板插值

In [2]:
variable_pool = {
    "sys": {"query": "低代码和 LangGraph 高代码的边界是什么？", "user_id": "u_1001"},
    "kb_1": {"hit_count": 1, "context": "[1] 低代码与高代码边界规范: 简单RAG用Dify，复杂状态机用LangGraph。"},
}

prompt_tpl = "用户({{sys.user_id}})提问：{{ sys.query }}\n命中条数：{{kb_1.hit_count}}\n参考资料：{{kb_1.context}}"
rendered = render_template(prompt_tpl, variable_pool)
print(rendered)
assert resolve_variable("kb_1.hit_count", variable_pool) == 1

用户(u_1001)提问：低代码和 LangGraph 高代码的边界是什么？
命中条数：1
参考资料：[1] 低代码与高代码边界规范: 简单RAG用Dify，复杂状态机用LangGraph。


## 3. DSL 解析与 Kahn 算法 DAG 环检测

In [3]:
dsl = build_default_rag_workflow_dsl()
definition = parse_workflow_dsl(dsl)
print(f"解析成功: {definition.name} | 节点数={len(definition.nodes)} | 边数={len(definition.edges)}")

# 验证非法环路会被拦截
cyclic_dsl = build_default_rag_workflow_dsl()
cyclic_dsl["edges"].append({"source": "llm_rag", "target": "kb_1"})
try:
    parse_workflow_dsl(cyclic_dsl)
except WorkflowValidationError as exc:
    print("成功拦截非法环路:", exc)

解析成功: 智枢企业知识库问答工作流 | 节点数=7 | 边数=6
成功拦截非法环路: DSL 工作流存在循环依赖（非合法 DAG）


## 4. 端到端执行：知识库命中分支 vs 未命中兜底分支

In [4]:
engine = WorkflowEngine()

res_hit = engine.run(dsl, inputs={"query": "MCP 协议怎么接入？"})
print("[命中分支] 轨迹:", [r.node_id for r in res_hit.execution_trace])
print("[命中分支] 路由:", res_hit.outputs["route"], "| hit_count:", res_hit.outputs["hit_count"])

res_miss = engine.run(json.dumps(dsl, ensure_ascii=False), inputs={"query": "今天北京天气怎么样？"})
print("[兜底分支] 轨迹:", [r.node_id for r in res_miss.execution_trace])
print("[兜底分支] 路由:", res_miss.outputs["route"], "| hit_count:", res_miss.outputs["hit_count"])

[命中分支] 轨迹: ['start', 'kb_1', 'cond_1', 'llm_rag', 'end_hit']
[命中分支] 路由: knowledge_rag | hit_count: 1
[兜底分支] 轨迹: ['start', 'kb_1', 'cond_1', 'llm_fallback', 'end_miss']
[兜底分支] 路由: fallback | hit_count: 0


## 5. 自定义检索器与大模型适配器注入

In [5]:
def custom_retriever(query: str, dataset_id: str, top_k: int) -> list[dict]:
    return [{"title": "星辰数科架构白皮书", "content": "LangGraph 负责复杂编排，DSL 引擎负责标准化流程。", "score": 0.98}]

def custom_llm(model: str, system_prompt: str, user_prompt: str, options: dict) -> str:
    return f"[自定义推理 {model}] 已收到问题与参考资料，核心结论：各司其职，低高代码协同。"

custom_engine = WorkflowEngine(retriever_fn=custom_retriever, llm_fn=custom_llm)
res_custom = custom_engine.run(dsl, inputs={"query": "智枢平台架构原则"})
print("自定义引擎输出:", res_custom.outputs["answer"])

自定义引擎输出: [自定义推理 deepseek-chat] 已收到问题与参考资料，核心结论：各司其职，低高代码协同。
